# Preliminaries

This notebook will show how to create, define, and run a ```Design``` object to perform sequence design of nucleic acids.

Following a generic introduction, example designs will 

## Importing the necessary classes

The ```Design``` class must be imported to build up a design:

In [1]:
%matplotlib inline
from matplotlib import pyplot as plt
import numpy as np

import sys
sys.path.insert(0, '/Users/Mark/build/nupack/api')
from nudev import *

The following instruction will assume that a design object called ```design``` has already been created.

# Specifying a design

## Physical parameters

A ```Design``` object has a property called model that can be overwritten with a new ```ModelSettings``` object to change the material, dangles/coaxial stacking model, temperature, and sodium and magnesium concentrations. 

```temperature``` is specified in Kelvin (K), and ion concentrations are specified in Molar (M).

Options for ```ensemble``` are:
* "nostacking": No dangle and coaxial stacking states in the expanded energy model are considered.
* "stacking" (the default): All dangle and coaxial stacking states in the expanded energy model are considered.
* "none": No dangle interactions are considered.
* "min": Unpaired nucleotides adjacent to base pairs always dangle stack. In cases were an unpaired nucleotide can dangle on either of two adjacent base pairs, the lower energy contribution is added
* "all": Unpaired nucleotides adjacent to base pairs always dangle stack. In cases were an unpaired nucleotide can dangle on either of two adjacent base pairs, the sum of the two contributions is added.

In [2]:
model = Model(material='rna', celsius=37, sodium=1.0, magnesium=0.0, ensemble='stacking')

## Specifying a domain

A domain is a set of consecutive nucleotides that appear as a subsequence of one or more strands in the design. Each domain is given a string name to allow reference to it when specifying strands. The method ```add_domain``` is used to define the domain in the design in terms of a string of [degenerate nucleotides codes](https://www.bioinformatics.org/sms/iupac.html). These strings are just Python strings, and as such can be constructed using string multiplication and concatenation, as shown in the following examples.

In [3]:
a = Domain('a', 'AAAA')
b = Domain('b', 'A'*4) # equivalent sequence specification

c = Domain('c', 'NNNNNNNNNN')
d = Domain('d', 'N'*10) # equivalent sequence specification

e = Domain('e', 'RRSSAAACCA')
f = Domain('f', 'R'*2 + 'S'*2 + 'A'*3 + 'C'*2 + 'A') # equivalent sequence specification

## Specifying a strand

Strands are contiguous (no nicks along the phosphate backbone) concatenations of domains; domains may appear in multiple strands or multiple times in the same strand. The method ```add_strand``` is used to define a strand by its name and a tuple/list of domain names.

In [4]:
A = Strand('A', [a, b, c])
B = Strand('B', [d, ~e])
C = Strand('C', [e, a, f])
D = Strand('D', [d, d, d])

## Specifying an on-target complex

On-target complexes are specified via the ```TargetComplex``` class. These complexes are specified by a name, a tuple/list of strands, and a string representing the target structure. The target structure is specified from $5^\prime$ to $3^\prime$ starting from the $5^\prime{-}$end of the first strand and ending at the $3^\prime{-}$end of the last strand.

Structure strings can be specified using one of three notations, with examples shown below:
* dot-parens-plus notation
* DU+ notation
* run-length encoded dot-parens-plus notation

In [5]:
# dot-parens-plus notation
C1 = TargetComplex('C1', [A, B, C], '........((((((((((+))))))))))((((((((((+))))))))))..............')

# DU+ notation
C2 = TargetComplex('C2', [D, D], 'D30 +')
C3 = TargetComplex('C3', [B, B, B], 'D10(D10 + D10 +)')
C4 = TargetComplex('C4', [B, A, B], 'D8(U12 +) D10(+) U10')

# run-length encoded dot-parens-plus notation
C5 = TargetComplex('C5', [B, C], '.10(10+)10.10')

## Specifying a test tube

A test tube ensemble is defined in two steps. First, the ```add_tube``` method is used to add a new tube to the design with a given name and a dictionary mapping the names of on-target complexes ($\Psi^\text{on}$) in the tube to their concentrations. Concentrations are always in units of Molar (M). Second, the method ```add_off_targets``` specifies the set of off-target complexes ($\Psi^\text{off}$) in the tube through three keywords, which can be combined arbitrarily:

* ```max_size```, integer in $[0,\infty]$, default=0: Adds all complexes of up to ```max_size``` composed of the strands of complexes in $\Psi^\text{on}$ to $\Psi^\text{off}$ such that $\Psi^\text{off} \cap \Psi^\text{on} = \varnothing$.
* ```include```, list of strings or iterables of strings, default=```None```: Each item in ```include``` is either a name of a complex, name of single strand, or iterable of names of strands. 
The implied complexes are added to $\Psi^\text{off}$, provided they are not already in $\Psi^\text{on}$.
* ```exclude```, list of strings or iterables of strings, default=```None```: Each item in ```exclude``` is either a name of a complex, name of single strand, or iterable of names of strands. 
The implied complexes are prevented from being added to $\Psi^\text{off}$ when processing ```max_size``` and ```include```.

In [6]:
# specify tubes by their names and on-target complexes with on-target concentrations
# specify named off-target complexes in tube
T1 = TargetTube('T1', {C1: 1e-6}, include=[C4, C5])

# specify unnamed off-targets each denoted by a strand ordering
T2 = TargetTube('T2', {C2: 1e-6}, include=[[D, D, D], [D, D, D, D]])

# specify combination of named and unnamed off-targets
T3 = TargetTube('T3', {C1: 0.000001, C2: 1e-3}, include=[C3, [A, A, B, B], [C], [D, D, D, D]])

# specify off-targets combinatorially:
# all complexes of up to 2 strands that are not on-targets in tube `T4'
T4 = TargetTube('T4', {C1: 2e-4, C3: 3e-5}, max_size=2)

# specify off-targets as the sum of sets
T5 = TargetTube('T5', {C4: 4e-6, C5: 5e-7}, include=[C3, [B]*4])

# specify off-targets as the difference of sets
T6 = TargetTube('T6', {C5: 6e-8}, max_size=3, exclude=[C3, [B] * 2])

## Specifying sequence constraints

### Match constraint

Match constraints are used to constrain concatenations of domains to be identical to each other. They are specified by providing the ```add_match_constraint``` with two lists of domains. The sum of the lengths of the domains in each list must be the same.

In [7]:
a = Domain('a','N10')
b = Domain('b','N4')
c = Domain('c','H6')
d = Domain('d','N6')
e = Domain('e','S2')

match1 = Match([c], [b, ~e])
match2 = Match([a, b], [d, d, e])

### Complementarity constraint

Complementarity constraints are used to constraint the concatenation of one list of domains to be the reverse complement of the concatenation of another list of domains. Therefore, the sum of the lenghts of the domains in each list must be the same.

Currently only Watson-Crick complementarity constraints are allowed. 

In addition to explicit domain list based specification of complementarity constraints, nucleotides that are base paired in the target structure of an on-target complex will have a complementarity constraint applied automatically once user specification is finished and the design algorithm begins.


In [8]:
comp = Complementarity([a, b], [c, d, e])

### Similarity constraint

A similarity constraint forces either a domain or strand to match a reference sequence of the same length at a number of positions that falls in a specified range. As such, the constraint is specified using

* the name of the domain or strand
* a reference sequence of the same length as the domain or strand
* a fractional range, $[l, u]$, where $0 \leq l < u \leq 1$

A common use case of the similarity constraint is to constrain a domain or strand to have GC content in a certain range. In this case, the reference sequence is just the degenerate base code ```S``` repeated for the length of the domain / strand.

In [9]:
a = Domain('a','N10')
sim = Similarity(a, 'S5K5', [0.25, 0.75])
    
# "composition constraint" special case: enforce 45-55% GC content
b = Domain('b', 'N20')
sim2 = Similarity(b, 'S20', [0.45, 0.55])

### Window constraint

A window constraint is used to constrain a concatenation of domains to have a sequence that is a substring of a given source sequence. 
It is specified in two steps. 
First, the source is defined by a name and a sequence. 
Then, the constraint itself is specified by giving the list of domains to concatenate and the name of the source sequence. 
The constraint can also allow the concatenated domains to have a sequence that is any window from several source sequences by giving a list of their names, instead of just one.

In [10]:
a = Domain('a', 'N'*10)
b = Domain('b', 'N'*10)
c = Domain('c', 'N'*10)
e = Domain('e', 'N'*10)

gfp = 'AUGGUGAGCAAGGGCGAGGAGCUGUUCACCGGGGUGGUGCCCAUCCUGGUCGAGCUGGACGGCGACGUAAACGGCCACAAGUUCAGCGUGUCCGGCGAGGGCGAGGGCGAUGCCACCUACGGCAAGCUGACCCUGAAGUUCAUCUGCACCACCGGCAAGCUGCCCGUGCCCUGGCCCACCCUCGUGACCACCCUGACCUACGGCGUGCAGUGCUUCAGCCGCUACCCCGACCACAUGAAGCAGCACGACUUCUUCAAGUCCGCCAUGCCCGAAGGCUACGUCCAGGAGCGCACCAUCUUCUUCAAGGACGACGGCAACUACAAG'

rfp = 'CCUGCAGGACGGCGAGUUCAUCUACAAGGUGAAGCUGCGCGGCACCAACUUCCCCUCCGACGGCCCCGUAAUGCAGAAGAAGACCAUGGGCUGGGAGGCCUCCUCCGAGCGGAUGUACCCCGAGGACGGCGCCCUGAAGGGCGAGAUCAAGCAGAGGCUGAAGCUGAAGGACGGCGGCCACUACGACGCUGAGGUCAAGACCACCUACAAGGCCAAGAAGCCCGUGCAGCUGCCCGGCGCCUACAACGUCAACAUCAAGUUGGACAUCACCUCCCACAACGAGGACUACACCAUCGUGGAACAGUACGAACGCGCCGAGGGCCGCCACUCCACCGGCGGCAUGGACGAGCUGUACAAGUAA'

# constrain window to be drawn from source
window1 = Window([a, ~b], [gfp])
# OR constrain window to be drawn from more than once source
window2 = Window([~c, e], [gfp, rfp])

### Library constraint

A library constraint forces a domain, or concatenated list of domains, to have its sequence come from a fixed set of enumerated sequences of the same length. The constraint is specified in two steps. First, one or more libraries are defined by giving them a name and a list of sequences, all of the same length for a given library. Then, the constraint itself is specified by giving a domain or list of domains and a library or list of libraries. The sum of the lengths of the domains must equal the sum of the library lengths. The library length is the length of any of its sequences.

In [11]:
a = Domain('a', 'N6')
b = Domain('b', 'N12')

# define a library of sequences
toeholds = ['CAGUGG', 'AGCUCG', 'CAGGGC']

# define a library of codons for each amino acid
aaI = ['AUU', 'AUC', 'AUA']
aaL = ['CUU', 'CUC', 'CUA', 'CUG', 'UUA', 'UUG']
aaV = ['GUU', 'GUC', 'GUA', 'GUG']
aaF = ['UUU', 'UUC']
aaM = ['AUG']
aaC = ['UGU', 'UGC']
aaA = ['GCU', 'GCC', 'GCA', 'GCG']
aaG = ['GGU', 'GGC', 'GGA', 'GGG']
aaP = ['CCU', 'CCC', 'CCA', 'CCG']
aaT = ['ACU', 'ACC', 'ACA', 'ACG']
aaS = ['UCU', 'UCC', 'UCA', 'UCG', 'AGU', 'AGC']
aaY = ['UAU', 'UAC']
aaW = ['UGG']
aaQ = ['CAA', 'CAG']
aaN = ['AAU', 'AAC']
aaH = ['CAU', 'CAC']
aaE = ['GAA', 'GAG']
aaD = ['GAU', 'GAC']
aaK = ['AAA', 'AAG']
aaR = ['CGU', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG']
aaSTOP = ['UAA', 'UAG', 'UGA']

# domain a is drawn from the `toeholds' library
lib1 = Library(a, toeholds)

# domain b is drawn from a concatenation of library sequences representing codons
lib2 = Library([b], [aaI, aaM, aaC, aaG])

### Pattern prevention constraint

Pattern prevention constraints are used to prevent any subsequences of a given strand or domain from containing some fixed pattern sequence. This pattern sequence can be specified using degenerate base codes.

Because this constraint is frequently applied with many patterns to many elements of the design, and possibly all strands with in a design, the method ```add_pattern_constraints``` allows specifying multiple pattern constraints simultaneously. The first (required) argument is a single pattern or list of patterns to be prevented. The second (keyword) argument, ```names```, has three valid specifications:

* ```None``` or unspecified: the patterns are prevented in all strands in the design
* a single domain or strand name: the patterns are prevented in only this domain or strand.
* a list of domain or strand names: the patterns are prevented in every named domain or strand.

In [12]:
a = Domain('a', 'N12')
b = Domain('b', 'N12')
A = Strand('A', [a, ~a])
B = Strand('B', [b, ~b])

# pattern prevention for a domain
pat1 = Pattern(['A4', 'U4'], where=[a])

# pattern prevention for a strand
pat2 = Pattern(['A4', 'U4'], where=[B])

# preventing the same patterns for strand `A' and domain `b'
pat3 = Pattern(['A5', 'C5', 'G5', 'U5'], where=[A, b])

# global pattern prevention
pat4 = Pattern(['A4', 'C4', 'G4', 'U4', 'M6', 'K6', 'W6', 'S6', 'R6', 'Y6'])

### Diversity constraint

New to NUPACK 4.0, diversity constraints represent a more efficient alternative to using pattern prevention constraints to ensure sequence diversity. 
For instance, specifying the constraints that no AAAA, CCCC, GGGG, or UUUU should appear in a strand is equivalent to specifying the constraint that every length 4 window of the strand must have at least 2 nucleotide constraints within. When specified as a diversity constraint, both the intention is more clear and the CSP solver is able to more rapidly make sequence mutations.

Diversity constraints are specified by two numbers:

* The first is the window length to consider for the strand(s)/domain(s).
* The second is the minimum number of nucleotide types that must appear in every window of the above length

Following are the two method calls necessary to reproduce the global pattern prevention above.

In [13]:
div1 = Diversity(4, 2)
div2 = Diversity(6, 3)

In the above examples, these diversity constraints are applied to all strands in the design.
Just like with pattern prevention constraints, diversity constraints can also be applied with one function call to a user-specified subset of domains or strands by adding a list of their names with the keyword ```where```.

In [14]:
div3 = Diversity(10, 4, where=[a, B])

If no objectives have been specified by the time running the design is requested, the design will add the multistate test tube ensemble defect automatically. The stop condition must still be set manually.

### Specifying pattern prevention soft constraint

Pattern prevention soft constraints are specified in nearly the same way as pattern prevention hard constraints.
The primary difference is that a weight can be supplied to control the relative design effort spent on the soft constraint.

In [15]:
pat = Pattern(patterns=['A4', 'C4', 'G4', 'U4', 'M6',
    'K6', 'W6', 'S6', 'R6', 'Y6'], weight=0.5)

### Specifying a similarity soft constraint

Similarity soft constraints are specified in nearly the same way as similarity hard constraints.
The primary difference is that a weight can be supplied to control the relative design effort spent on the soft constraint.

In [16]:
a = Domain('a','N10')
b = Domain('b','N20')

# explicitly specify weight
sim = Similarity('b', 'S20', [0.45, 0.55], weight=0.25)

### Specifying a sequence symmetry soft constraint

Sequence symmetry soft constraints are specified with a list of complex names (or single complex name) to consider simultaneously.
This will penalize windows (i.e. n-grams, critons) that repeat spuriously (not explicitly constrained to be identical) and reverse complement windows that are not in full duplex regions.
Multiple sequence symmetry constraints with different window sizes can be specified for the same sets of complexes, as shown below.

In [17]:
a = Domain('a', 'N12')
b = Domain('b', 'N12')
A = Strand('A', [a, ~a])
B = Strand('B', [b, ~b])

C = TargetComplex('C', [A], "(10.4)10")
D = TargetComplex('D', [A, A], "D12 +")

ssm1 = SSM([C, D], 4, weight=0.15)

# the same complexes with larger windows weighted higher
ssm2 = SSM([C, D], 5, weight=0.25)
ssm3 = SSM([C, D], 6, weight=0.45)

### Specifying a duplex structure energy equalization constraint

Currently, the only structural motif that can be equalized is a perfect duplex.
This is specified by giving a list of domain names.
The soft constraint will then bias search toward sequences that for each domain ```a```, the duplex with complementary domain ```a*``` will approach the median of all the constrained duplexes.
A fixed reference energy can also be supplied through the ```energy``` keyword argument, which will try to force the duplex free energies to match that reference energy instead.

In [18]:
# equalize to median value
diff1 = EnergyDifference([a, b])

# equalize to reference value, with explicit weight
diff2 = EnergyDifference([a, b], energy=-17, weight=0.5)

## Specifying weights

The user may wish to alter the relative weighting of defect contributions within the design objective function, $\mathcal{M}$, to prioritize or deprioritize design quality for a portion of the design ensemble. Custom defect weights can be defined for any level within the design ensemble (tube, complex, strand, domain), or for any combination of levels (specified coarser to finer with a period separating each level). Each weight takes a value in the interval $[0,\infty)$. By default, all weights are unity. Increasing the weight for a tube, complex, strand or domain will lead to a corresponding increase in the allocation of effort to designing this entity, typically leading to a corresponding reduction in the defect contribution of the entity. Likewise, decreasing the weight for a tube, complex, strand or domain will lead to a corresponding decrease in the allocation of effort to designing this entity, typically leading to a corresponding increase in the defect contribution of the entity. Weights specified at multiple levels within the ensemble are multiplicative (see Supplementary Information of the [multistate design paper](https://pubs.acs.org/doi/10.1021/jacs.6b12693) for details). With the default value of unity for all weights, $\mathcal{M}$ reduces to the multistate test tube ensemble defect, representing the average equilibrium fraction of incorrectly paired nucleotides over the design ensemble. With custom weights, the physical meaning of the objective function is distorted in the service of adjusting design priorities. The following script illustrates assignment of defect weights at different levels within the design ensemble:    
  

In [19]:
a = Domain('a', 'N10')
b = Domain('b', 'N10')

A = Strand('A', [a, b])
B = Strand('B', [~b, ~a])

AB = TargetComplex('A+B', [A, B], structure='(20+)20')
AA = TargetComplex('A+A', [A, A], structure='(20+)20')
t1 = TargetTube('t1', {AB: 1e-8}, max_size=2)
t2 = TargetTube('t2', {AA: 1e-8, AB: 1e-8}, max_size=2)

tubes = [t1, t2]
weights = Weights(tubes) # All weights are initialized to 1

weights[:, :, :, a] *= 2
weights[:, :, A] = 4
weights[t2] = 2
weights[t1, AB] = 5,
weights[:, :, A, b] = 0.75
weights[t2, AA, D, a] = 0.5
weights[t2, :, :, d] = 3

weights

In [20]:
weights[t2]

0    2.00
1    2.00
2    0.75
3    2.00
4    0.75
5    2.00
Name: weight, dtype: float64

## Specifying algorithm parameters

The default design parameters are shown below. They can be changed by assigning the named attribute.

In [21]:
params = design.Parameters(M_reopt=1, stop=0.02)

In addition to the multistate test tube design algorithm parameters, a few others are included in the ```DesignParameters``` object:

* ```seed```: The seed for the random number generator allowing reproducible design runs
* ```cache_bytes_of_RAM```: The number of bytes of RAM to set as a maximum cache size for thermodynamic block caching
* ```min_ppair```: The minimum pair probability used as a threshhold for converting dense pair probability matrices into sparse representation for efficiency
* ```slowdown```: For development purposes. Runs all thermodynamics calls ```slowdown``` times instead of once.
* ```time_analysis```: A boolean that determines whether the full ensemble is reevaluated at the end to get an accurate timing of the cost of analysis. Can be set to ```False``` to speed up design output.

The remaining log options are all strings and can have four different states:

* ```""```: The empty string (default). If empty, no logging of this type occurs.
* ```"stdout"```: The log is written to the standard output stream.
* ```"stderr"```: The log is written to the standard error stream.
* any other string: The string is treated as a relative path specification where intermediate folders must exist and the log is written to the file at that path.

The information logged for each of these given a non-empty string is as follows:

* ```log```: After every major algorithm component finishes, time since design start, sequence, defect (estimate), algorithm position, decomposition tree position, and active/passive ensemble breakdown are logged.
* ```decomposition_log```: After each time a complex (on- or off-target) is decomposed (or redecomposed), a JSON representation of the decomposition tree is logged.
* ```thermo_log```: Primarily for debugging/development purposes. After every thermodynamic evaluation, the type of calculation (pair probability, bonused pair probability, partition function), number of nucleotides evaluated, and duration of the calculation are logged.

## Running the design

Running a design is done by using the ```Design``` object's function call operator, e.g. ```design()```. 

In [22]:
a = Domain('a', 'N20')

A = Strand('A', [a])
B = Strand('B', [~a])

C = TargetComplex('C', [A, B], '(20+)20')

tube = TargetTube('tube1', {C: 1e-6}, max_size=2)

result = tube_design([tube], model=Model())

result

Result(mapping=Mapping(tubes={Tube('tube1', strands={Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),)): 1e-06, Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),)): 1e-06}, complexes=[Complex([Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]), Complex([Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),)), Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]), Complex('C', [Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),)), Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]), Complex([Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]), Complex([Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),)), Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),))])]): Tube('tube1', strands={Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)): 1e-08, Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),)): 1e-08}, complexes=[Complex('[A]', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),))]), Complex('[A+A]', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)), Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),))]), Complex('C', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)), Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))]), Complex('[B]', [Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))]), Complex('[B+B]', [Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),)), Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))])])}, complexes={Complex([Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),)), Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]): Complex('[A+A]', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)), Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),))]), Complex([Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),)), Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]): Complex('[B+B]', [Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),)), Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))]), Complex([Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]): Complex('[B]', [Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))]), Complex('C', [Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),)), Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]): Complex('C', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)), Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))]), Complex([Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),))]): Complex('[A]', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),))])}, strands={Strand('A', (Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')),)): Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)), Strand('B', (Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')),)): Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))}, domains={Domain('a', Sequence('NNNNNNNNNNNNNNNNNNNN')): Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')), Domain('a*', Sequence('NNNNNNNNNNNNNNNNNNNN')): Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC'))}), defects=<nudev.design.Defects object at 0x7fadfd3ac1f0>, analysis=Result(tubes={Tube('tube1', strands={Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)): 1e-08, Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),)): 1e-08}, complexes=[Complex('[A]', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),))]), Complex('[A+A]', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)), Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),))]), Complex('C', [Strand('A', (Domain('a', Sequence('GAGCCTCTGTTCATTCACCC')),)), Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))]), Complex('[B]', [Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),))]), Complex('[B+B]', [Strand('B', (Domain('a*', Sequence('GGGTGAATGAACAGAGGCTC')),)), Strand('B'

# Examples

The following cell imports functions from packages necessary for bokeh visualization of design progress and design results. The function ```run_and_display``` runs the design in a separate thread to allow visualization of design progress in the notebook during design. The function ```notebook_results``` takes a ```DesignResult``` object, saves it to a file (temporary or user specified), and the loads this file into an interactive panel for exploring design defects.

In [23]:
# from nupack.visualize import run_and_display
# from nupack.residuals import notebook_results

## Design Evaluation Example (Example 7 from NUPACK 3.2 User Guide)

In [24]:
# set physical parameters
model = Model(material='rna', celsius=37)

# define domains
a = Domain('a', 'N6')
c = Domain('c', 'N8')
b = Domain('b', 'N4')
w = Domain('w', 'N2')
y = Domain('y', 'N4')
x = Domain('x', 'N12')
z = Domain('z', 'N3')
s = Domain('s', 'N5')

# define strands from domains
Cout_s   = Strand('Cout_s', [w, x, y, s])
A_s      = Strand('A_s', [~c, ~b, ~a, ~z, ~y])
A_toe_s  = Strand('A_toe_s', [~c])
C_s      = Strand('C_s', [w, x, y, s, ~a, ~z, ~y, ~x, ~w])
C_loop_s = Strand('C_loop_s', [s, ~a, ~z])
B_s      = Strand('B_s', [x, y, z, a, b])
Xs_s     = Strand('Xs_s', [a, b, c])

# define complexes composed of one or more strands in a given order AND
# define target structures for each complex
C      = TargetComplex('C', [C_s], 'D2 D12 D4( U5 U6 U3 )')
B      = TargetComplex('B', [B_s], 'U12 U4 U3 U6 U4')
C_loop = TargetComplex('C_loop', [C_loop_s], 'U14')
A_B    = TargetComplex('A_B', [A_s, B_s], 'U8 D4 D6 D3 D4(+ U12)')
X      = TargetComplex('X', [Xs_s], 'U18')
X_A    = TargetComplex('X_A', [Xs_s, A_s], 'D6 D4 D8(+) U3 U4')
C_out  = TargetComplex('C_out', [Cout_s], 'U23')
B_C    = TargetComplex('B_C', [B_s, C_s], 'D12 D4 D3 D6 (U4 + U2 U12 U4 U5) U2')
A_toe  = TargetComplex('A_toe', [A_toe_s], 'U8')

# on-target tubes
Step_0 = TargetTube('Step_0', {C: 1e-08, X: 1e-08, A_B: 1e-08}, max_size=2, include=[[A_s], [B_s]], exclude=[X_A])

Step_1 = TargetTube('Step_1', {X_A: 1e-08, B: 1e-08}, max_size=2, include=[X, A_B])

Step_2 = TargetTube('Step_2', {B_C: 1e-08}, max_size=2, include=[B, C])

# global orthogonality tube
Crosstalk = TargetTube('Crosstalk', {
    A_B: 1e-08,
    C: 1e-08,
    X: 1e-08,
    B: 1e-08,
    C_out: 1e-08,
    C_loop: 1e-08,
    A_toe: 1e-08,
}, max_size=2, exclude=[X_A, B_C, [Xs_s, A_toe_s], [B_s, C_loop_s]])

# GC content constraints
hard = [
    Similarity(d, 'S'*len(d), (0.45, 0.55)) 
        for d in [Cout_s, A_s, C_s, C_loop_s, B_s, Xs_s]
]

# sources lines
tpm3 = 'GAACACTATTAGCTATTTGTAGTACTCTAAAGAGGACTGCAGAACGCATCGCAGTAGTGGTGAAAAGCCGTGCGTGCGCGTGAAACATCTGATCCTCACGTTACTTCCACTCGCTCTGCGTTTGACTTGTTGGCGGGGCGTTGGTGCCTTGGACTTTTTTTTCCTCCTTCTCTTCTTCGCGGCTCGGTCCACTACGCTGCTCGAGAGGAATCTGCTTTATTCGACCACACTACTCCTAAAGTAACACATTAAAATGGCCGGATCAAACAGCATCGATGCAGTTAAGAGAAAAATCAAAGTTTTACAACAGCAAGCAGATGAGGCAGAAGAAAGAGCCGAGATTTTGCAGAGACAGGTCGAGGAGGAGAAGCGTGCCAGGGAGCAGGCTGAGGCAGAGGTGGCTTCTCTGAACAGGCGTATCCAGCTGGTTGAGGAGGAGTTGGATCGTGCTCAGGAGAGACTGGCCACAGCCCTGCAAAAGCTGGAGGAAGCCGAGAAGGCCGCAGATGAGAGCGAGAGAGGGATGAAGGTGATTGAGAACAGGGCTCTGAAGGATGAGGAGAAGATGGAGCTGCAGGAGATCCAGCTTAAGGAGGCCAA'
hard += [Window([a, b, c], tpm3)]

desm = 'CATTTACACAGCGTACAAACCCAACAGGCCCAGTCATGAGCACGAAATATTCAGCCTCCGCCGAGTCGGCGTCCTCTTACCGCCGCACCTTTGGCTCAGGTTTGGGCTCCTCTATTTTCGCCGGCCACGGTTCCTCAGGTTCCTCTGGCTCCTCAAGACTGACCTCCAGAGTTTACGAGGTGACCAAGAGCTCCGCTTCTCCCCATTTTTCCAGCCACCGTGCGTCCGGCTCTTTCGGAGGTGGCTCGGTGGTCCGTTCCTACGCTGGCCTTGGTGAGAAGCTGGATTTCAATCTGGCTGATGCCATAAACCAGGACTTCCTCAACACGCGTACTAATGAGAAGGCCGAGCTCCAGCACCTCAATGACCGCTTCGCCAGCTACATCGAGAAGGTGCGCTTCCTCGAGCAGCAGAACTCTGCCCTGACGGTGGAGATTGAGCGTCTGCGGGGTCGCGAGCCCACCCGTATTGCAGAGCTGTACGAGGAGGAGATGAGAGAGCTGCGCGGACAGGTGGAGGCACTGACCAATCAGAGATCCCGTGTGGAGATCGAGAGGGACAACCTAGTCGATGACCTACAGAAACTAAAGCTCAGACTTC'
hard += [Window([w, x, y, z], desm)]

# Prevented patterns
hard += [Pattern(['A4','C4','G4','U4'])]

params = design.Parameters(seed=93, stop=0.1)
result = tube_design([Step_0, Step_0, Step_0, Crosstalk], model=Model(), hard_constraints=[] and hard, parameters=params)

## Design Evaluation Example (Example 8 from NUPACK 3.2 User Guide)

In [25]:
# set physical properties
model = Model(material='RNA', celsius=23)

# define domains
a = Domain('a', 'ACCUCCAAGCACAACUGUGGCCCCAUA')
b = Domain('b', 'GGGGCCGGAUUACAACUUUCCCUGUGAAC')
c = Domain('c', 'AUCACAGACAGUUAACCACUUGAGG')
d = Domain('d', 'AUCAAGUGGGCUUGGAGC')

# define strands from domains
left = Strand('left', [a])
top = Strand('top', [b])
right = Strand('right', [c])
bottom = Strand('bottom', [d])

# define complex compsed of strands in a given order AND
# Define target structure for complex
stickfigure = TargetComplex('stickfigure', (left, top, right, bottom), "U2D8(U2D6(D6(U3+)D3U9D6(U2+U1))U2D8(U2+U1))U1")

# define test tube
figuretube = TargetTube('figuretube', {stickfigure: 1e-6}, max_size=3)

# evaluate
result = tube_evaluate([figuretube], model)

In [26]:
from nudev.design import Design
import pickle
des = Design([figuretube], model=Model())
s = pickle.dumps(des)

# Saving and restarting a Design

When "calling" the design to start the optimization process, two additional arguments must be added for checkpointing to work, `checkpoint_condition` and `checkpoint_handler`.

`checkpoint_condition` is a binary function that receives the stats and timer object from the C++ `Designer` object after steps in the design. The logic in `checkpoint_condition` then uses this information to determine whether a checkpoint should be made, in which case it returns True. In the call below, it is set to an object of an included class, `TimeInterval`. If `checkpoint_condition` is set to an object `TimeInterval(n)`, then a checkpoint will be emitted roughly every n seconds.

`checkpoint_handler` is the function which actually does something given that `checkpoint_condition` returns `True`. `checkpoint_handler` takes one argument, a Result object, and decides how it will use this information. In the call below, it is set to an object of the included class, `WriteToFileCheckpoint`. This type of `checkpoint_handler` object is instantiated with a filename prefix ("design_test" below) and will convert the design Result object into JSON and serialize it to a file with the given prefix and a time stamp, e.g. design_test-2020-01-27T00:16:52.170292.out

In [27]:
from nudev.design import TimeInterval, WriteToFileCheckpoint 

result = des.optimize(checkpoint_condition=TimeInterval(1), checkpoint_handler=WriteToFileCheckpoint("design-checkpoint"))

## Saving the final design outputs in a text file

In [28]:
result.save('design-result.o')
result2 = design.Result.load('design-result.o')

## Running from a checkpoint file
The following lines of code will run a design using the final output as a checkpoint file. The argument restart must be a python design `Result` object, in this case loaded from a file containing a valid JSON representation of a `Result` object.

In [29]:
newer_result = des.optimize(restart=result2)